In [102]:
# https://github.com/arpanghosh8453/programs/blob/master/Fitbit%20Data%20Analyzer/Fitbit%20HR%20analyzer.ipynb
# https://dev.fitbit.com/
# https://dev.fitbit.com/build/reference/web-api/basics/
# Implicit Grant Flow
# You need the following modules, if you don't have them, use pip install <module-name>

import requests
import pandas as pd
import datetime
import numpy as np
import json

import dash
import dash_core_components as dcc
import dash_html_components as html
import plotly.express as px
import jupyter_dash
from dash.dependencies import Input, Output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']

In [103]:
with open("data/credentials.json", "r") as file:
    credentials = json.load(file)
    fitbit_cr = credentials['fitbit_cr']
    fitbit_key = fitbit_cr['KEY']
    username = fitbit_cr['USERNAME']

In [104]:
# day_range_length = 1

# start_date = str((datetime.datetime.now() - datetime.timedelta(days=day_range_length)).date())
# #start_date = "2020-01-01"

# end_date = str(datetime.datetime.date(datetime.datetime.now())) 
# #end_date = "2020-01-02"

### Date range

In [135]:
start_date = '2020-01-01'
end_date = '2020-02-01'
print(start_date, "to", end_date)

2020-01-01 to 2020-02-01


In [136]:
#Update your start and end dates here in yyyy-mm-dd format 
start = datetime.datetime.strptime(start_date, "%Y-%m-%d")
end = datetime.datetime.strptime(end_date, "%Y-%m-%d")

date_array = (start + datetime.timedelta(days=x) for x in range(0, (end-start).days))

day_list = []
for date_object in date_array:
    day_list.append(date_object.strftime("%Y-%m-%d"))
    
print("Date Range : ", day_list)

Date Range :  ['2020-01-01', '2020-01-02', '2020-01-03', '2020-01-04', '2020-01-05', '2020-01-06', '2020-01-07', '2020-01-08', '2020-01-09', '2020-01-10', '2020-01-11', '2020-01-12', '2020-01-13', '2020-01-14', '2020-01-15', '2020-01-16', '2020-01-17', '2020-01-18', '2020-01-19', '2020-01-20', '2020-01-21', '2020-01-22', '2020-01-23', '2020-01-24', '2020-01-25', '2020-01-26', '2020-01-27', '2020-01-28', '2020-01-29', '2020-01-30', '2020-01-31']


In [87]:
# df_all = pd.DataFrame()

In [88]:
hr_df = pd.DataFrame()

### Heart Rate requests

In [81]:
hr_response = requests.get("https://api.fitbit.com/1/user/-/activities/heart/date/"+ single_day +"/1d/1min/time/00:00/23:59.json", headers=header).status_code
hr_response

200

In [89]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

for single_day in day_list:
    hr_response = requests.get("https://api.fitbit.com/1/user/-/activities/heart/date/"+ single_day +"/1d/1min/time/00:00/23:59.json", headers=header).json()
    try:
        df = pd.DataFrame(hr_response['activities-heart-intraday']['dataset'])
        date = pd.Timestamp(single_day).strftime('%Y-%m-%d')
        df = df.set_index(pd.to_datetime(date + ' ' + df['time'].astype(str)))
        #print(df)
        hr_df = hr_df.append(df, sort=True)
    except KeyError as e:
        print("No data for the given date", date)
    
# df_all.index.set_names('dateTime', inplace = True)   
# del df_all['time']
hr_df

,time,value
time,,
2020-01-01 00:00:00,00:00:00,57
2020-01-01 00:01:00,00:01:00,57
2020-01-01 00:02:00,00:02:00,56
2020-01-01 00:03:00,00:03:00,56
2020-01-01 00:04:00,00:04:00,57
...,...,...
2020-01-31 23:55:00,23:55:00,66
2020-01-31 23:56:00,23:56:00,62
2020-01-31 23:57:00,23:57:00,60


In [90]:
del hr_df['time']
hr_df

,value
time,
2020-01-01 00:00:00,57
2020-01-01 00:01:00,57
2020-01-01 00:02:00,56
2020-01-01 00:03:00,56
2020-01-01 00:04:00,57
...,...
2020-01-31 23:55:00,66
2020-01-31 23:56:00,62
2020-01-31 23:57:00,60


In [91]:
# #Put the interval you want to take the average of the imported data from fitbit with 2-5 sec interval. Default 10 minute
# summary_df = (df_all['value'].resample('10Min').mean())
# summary_df

In [92]:
# # trying to create empty <None> list so that gaps show up in the heart rate dashboard when I don't have it on
# blank_range = pd.DataFrame()
# blank_range['time'] = pd.date_range("1/1/2020", "1/8/2020", freq="T")
# #blank_range['time'].to_string()
# blank_range['value'] = None
# blank_range

In [93]:
# hr_df = pd.read_csv('~/qs/fitbit/data/hr.csv')
# hr_df

In [94]:
print(hr_df['time'][0])
type(hr_df['time'][0])

KeyError: 'time'

In [95]:
hr_df['time'] = pd.to_datetime(hr_df['time'])
type(hr_df['time'][0])

KeyError: 'time'

In [96]:
hr_df = hr_df.set_index('time')
hr_df

KeyError: "None of ['time'] are in the columns"

#### How to set the missing timestamp values as None

In [97]:
# https://stackoverflow.com/questions/51656065/pandas-resampling-typeerror-only-valid-with-datetimeindex-timedeltaindex-or-p
# https://stackoverflow.com/questions/54592536/find-gaps-in-pandas-time-series-dataframe-sampled-at-1-minute-intervals-and-fill
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.resample.html
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.asfreq.html#pandas.DataFrame.asfreq
hr_df = hr_df.resample('T', convention='start').asfreq()
hr_df

,value
time,
2020-01-01 00:00:00,57.0
2020-01-01 00:01:00,57.0
2020-01-01 00:02:00,56.0
2020-01-01 00:03:00,56.0
2020-01-01 00:04:00,57.0
...,...
2020-01-31 23:55:00,66.0
2020-01-31 23:56:00,62.0
2020-01-31 23:57:00,60.0


In [98]:
hr_df['time'] = hr_df.index
hr_df

,value,time
time,,
2020-01-01 00:00:00,57.0,2020-01-01 00:00:00
2020-01-01 00:01:00,57.0,2020-01-01 00:01:00
2020-01-01 00:02:00,56.0,2020-01-01 00:02:00
2020-01-01 00:03:00,56.0,2020-01-01 00:03:00
2020-01-01 00:04:00,57.0,2020-01-01 00:04:00
...,...,...
2020-01-31 23:55:00,66.0,2020-01-31 23:55:00
2020-01-31 23:56:00,62.0,2020-01-31 23:56:00
2020-01-31 23:57:00,60.0,2020-01-31 23:57:00


In [99]:
# from IPython.display import display
# with pd.option_context('display.max_rows', 10500):
#     display(df2)

In [100]:
# import matplotlib.pyplot as plt

# plt.rcParams["figure.figsize"]=20,8

# # Heart rate data summary [10min avg] from start date[2021-03-18] to end date[2021-03-21] 
# #if you are using matplotlib directly in python ( py file ) then use plt.plot(summary_df,kind='area')
# summary_df.plot.area(ylim=(40,160))

In [101]:
hr_df.to_csv('data/hr.csv', index=None, encoding='utf-8')

### Resting Heart Rate requests

In [121]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

resting_hr_response = requests.get("https://api.fitbit.com/1/user/-/activities/heart/date/"+start_date+"/"+end_date+".json", headers=header).json()

In [123]:
# resting_hr_response

In [25]:
#all_resting_HR_list = []
for i in resting_hr_response['activities-heart']:
    try:
        resting_dict = { 'dateTime':i['dateTime'], "resting_HR":i['value']['restingHeartRate']}
        all_resting_HR_list.append(resting_dict)
    except KeyError as e:
        print("No data for the given date", i['dateTime'])
    
resting_HR_df = pd.DataFrame(all_resting_HR_list)
resting_HR_df['time'] = resting_HR_df.dateTime.apply(pd.Timestamp)
 
# resting_HR_df.set_index("dateTime", inplace = True)
# resting_HR_df['time'] = resting_HR_df.index

In [27]:
del resting_HR_df['dateTime']
resting_HR_df

,resting_HR,time
0,56,2020-01-01
1,58,2020-01-02
2,57,2020-01-03
3,58,2020-01-04
4,57,2020-01-05
5,58,2020-01-06
6,59,2020-01-07
7,58,2020-01-08
8,56,2020-01-01
9,58,2020-01-02


### Step Count

In [12]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

step_response = requests.get("https://api.fitbit.com/1/user/-/activities/steps/date/"+start_date+"/"+end_date+"/1min.json", headers=header).json()['activities-steps']

In [13]:
step_df = pd.DataFrame(step_response)
step_df['time'] = pd.to_datetime(step_df['dateTime'].apply(pd.Timestamp)).dt.date
del step_df['dateTime']
step_df['Step Count'] = step_df['value'].apply(int)
del step_df['value']
step_df

,time,Step Count
0,2020-01-01,1041
1,2020-01-02,321
2,2020-01-03,1659
3,2020-01-04,8407
4,2020-01-05,4520
...,...,...
361,2020-12-27,4143
362,2020-12-28,5340
363,2020-12-29,3503
364,2020-12-30,6311


In [14]:
step_df.to_csv('data/steps.csv', index=None, encoding='utf-8')

### Distance

In [124]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

distance_response = requests.get("https://api.fitbit.com/1/user/-/activities/distance/date/"+start_date+"/"+end_date+"/1min.json", headers=header).json()['activities-distance']

In [125]:
distance_df = pd.DataFrame(distance_response)
distance_df['time'] = pd.to_datetime(distance_df['dateTime'].apply(pd.Timestamp)).dt.date
del distance_df['dateTime']
distance_df['Distance in KM'] = distance_df['value'].apply(float)
del distance_df['value']
distance_df

,time,Distance in KM
0,2020-01-01,0.82576
1,2020-01-02,0.24246
2,2020-01-03,1.25603
3,2020-01-04,6.37055
4,2020-01-05,3.49040
...,...,...
362,2020-12-28,4.04441
363,2020-12-29,2.68446
364,2020-12-30,4.77629
365,2020-12-31,2.75618


In [126]:
distance_df['Distance in MI'] = distance_df['Distance in KM'].apply(lambda x: x * 0.62137)
distance_df

In [128]:
distance_df.to_csv('data/distance.csv', index=None, encoding='utf-8')

### Sleep

In [208]:
header = {'Authorization': 'Bearer {}'.format(fitbit_key)}

sleep_response = requests.get("https://api.fitbit.com/1.2/user/-/sleep/date/"+start_date+"/"+end_date+".json", headers=header).status_code
sleep_response

200

In [209]:
sleep_response = requests.get("https://api.fitbit.com/1.2/user/-/sleep/date/"+start_date+"/"+end_date+".json", headers=header).json()

In [210]:
sleep_duration_list = []
for i in sleep_response['sleep']:
    sleep_duration_list.append({'date':i['dateOfSleep'], 'start_time':i['startTime'], 'end_time':i['endTime']})
    
sleep_df = pd.DataFrame(sleep_duration_list)

In [211]:
sleep_df

,date,start_time,end_time
0,2020-02-01,2020-02-01T11:40:00.000,2020-02-01T14:47:30.000
1,2020-01-31,2020-01-31T11:24:00.000,2020-01-31T18:07:30.000
2,2020-01-31,2020-01-30T20:55:00.000,2020-01-31T04:53:00.000
3,2020-01-30,2020-01-29T12:59:00.000,2020-01-30T03:08:00.000
4,2020-01-28,2020-01-28T10:18:00.000,2020-01-28T18:01:30.000
5,2020-01-27,2020-01-27T10:24:30.000,2020-01-27T18:06:30.000
6,2020-01-26,2020-01-26T14:07:00.000,2020-01-26T18:30:30.000
7,2020-01-26,2020-01-25T21:44:00.000,2020-01-26T07:49:30.000
8,2020-01-25,2020-01-24T20:08:00.000,2020-01-25T03:35:00.000
9,2020-01-23,2020-01-23T11:50:30.000,2020-01-23T17:59:00.000


In [212]:
type(sleep_df['start_time'][0])

str

In [213]:
# https://stackoverflow.com/questions/62672427/how-do-i-convert-iso-8601-object-into-datetime-format-in-a-pandas-dataframe
sleep_df['start_time'] = pd.to_datetime(sleep_df['start_time'])
sleep_df['end_time'] = pd.to_datetime(sleep_df['end_time'])
sleep_df

,date,start_time,end_time
0,2020-02-01,2020-02-01 11:40:00,2020-02-01 14:47:30
1,2020-01-31,2020-01-31 11:24:00,2020-01-31 18:07:30
2,2020-01-31,2020-01-30 20:55:00,2020-01-31 04:53:00
3,2020-01-30,2020-01-29 12:59:00,2020-01-30 03:08:00
4,2020-01-28,2020-01-28 10:18:00,2020-01-28 18:01:30
5,2020-01-27,2020-01-27 10:24:30,2020-01-27 18:06:30
6,2020-01-26,2020-01-26 14:07:00,2020-01-26 18:30:30
7,2020-01-26,2020-01-25 21:44:00,2020-01-26 07:49:30
8,2020-01-25,2020-01-24 20:08:00,2020-01-25 03:35:00
9,2020-01-23,2020-01-23 11:50:30,2020-01-23 17:59:00


In [214]:
sleep_df['Start'] = sleep_df['start_time']
sleep_df['End'] = sleep_df['end_time']
sleep_df['Day'] = sleep_df['date']

del sleep_df['start_time']
del sleep_df['end_time']
del sleep_df['date']

sleep_df

,Start,End,Day
0,2020-02-01 11:40:00,2020-02-01 14:47:30,2020-02-01
1,2020-01-31 11:24:00,2020-01-31 18:07:30,2020-01-31
2,2020-01-30 20:55:00,2020-01-31 04:53:00,2020-01-31
3,2020-01-29 12:59:00,2020-01-30 03:08:00,2020-01-30
4,2020-01-28 10:18:00,2020-01-28 18:01:30,2020-01-28
5,2020-01-27 10:24:30,2020-01-27 18:06:30,2020-01-27
6,2020-01-26 14:07:00,2020-01-26 18:30:30,2020-01-26
7,2020-01-25 21:44:00,2020-01-26 07:49:30,2020-01-26
8,2020-01-24 20:08:00,2020-01-25 03:35:00,2020-01-25
9,2020-01-23 11:50:30,2020-01-23 17:59:00,2020-01-23


In [215]:
df = sleep_df

In [216]:
df['Day'] = pd.to_datetime(df.Day)

# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.dt.html
df["Day"] = pd.to_datetime(df["Day"]).dt.dayofyear

day = df['Day']
day

0     32
1     31
2     31
3     30
4     28
5     27
6     26
7     26
8     25
9     23
10    22
11    21
12    20
13    19
14    18
15    17
16    16
17    15
18    14
19    13
20    12
21    11
22    10
23     9
24     8
25     7
26     6
27     5
28     4
29     3
30     3
31     2
32     1
Name: Day, dtype: int64

In [217]:
df['Start'] = pd.to_datetime(df.Start)
df["Start"] = df["Start"].apply(lambda x: x.replace(year=1970, month=1, day=1))

In [218]:
df['End'] = pd.to_datetime(df.End)
df["End"] = df["End"].apply(lambda x: x.replace(year=1970, month=1, day=1))

In [219]:
df

,Start,End,Day
0,1970-01-01 11:40:00,1970-01-01 14:47:30,32
1,1970-01-01 11:24:00,1970-01-01 18:07:30,31
2,1970-01-01 20:55:00,1970-01-01 04:53:00,31
3,1970-01-01 12:59:00,1970-01-01 03:08:00,30
4,1970-01-01 10:18:00,1970-01-01 18:01:30,28
5,1970-01-01 10:24:30,1970-01-01 18:06:30,27
6,1970-01-01 14:07:00,1970-01-01 18:30:30,26
7,1970-01-01 21:44:00,1970-01-01 07:49:30,26
8,1970-01-01 20:08:00,1970-01-01 03:35:00,25
9,1970-01-01 11:50:30,1970-01-01 17:59:00,23


In [220]:
def create_dict(df):
    dictionary = []
    for row in df.index:
        start = str(f"'{df.loc[row].at['Start']}'")
        end = str(f"'{df.loc[row].at['End']}'")
        day = str(f"'{df.loc[row].at['Day']}'")
        #activity = str(f"'{df.loc[row].at['Activity type']}'")
        dictionary.append(f'dict(Start={start}, End={end}, Day={day}),')
    return dictionary

In [221]:
new_df = create_dict(df)
def new_dictionary(new_df):
    dictionary = []
    for row in new_df:
        dictionary = print(row)
    return dictionary

In [222]:
new_dictionary(new_df)

dict(Start='1970-01-01 11:40:00', End='1970-01-01 14:47:30', Day='32'),
dict(Start='1970-01-01 11:24:00', End='1970-01-01 18:07:30', Day='31'),
dict(Start='1970-01-01 20:55:00', End='1970-01-01 04:53:00', Day='31'),
dict(Start='1970-01-01 12:59:00', End='1970-01-01 03:08:00', Day='30'),
dict(Start='1970-01-01 10:18:00', End='1970-01-01 18:01:30', Day='28'),
dict(Start='1970-01-01 10:24:30', End='1970-01-01 18:06:30', Day='27'),
dict(Start='1970-01-01 14:07:00', End='1970-01-01 18:30:30', Day='26'),
dict(Start='1970-01-01 21:44:00', End='1970-01-01 07:49:30', Day='26'),
dict(Start='1970-01-01 20:08:00', End='1970-01-01 03:35:00', Day='25'),
dict(Start='1970-01-01 11:50:30', End='1970-01-01 17:59:00', Day='23'),
dict(Start='1970-01-01 12:17:30', End='1970-01-01 21:17:30', Day='22'),
dict(Start='1970-01-01 02:18:30', End='1970-01-01 16:38:30', Day='21'),
dict(Start='1970-01-01 09:18:30', End='1970-01-01 18:37:30', Day='20'),
dict(Start='1970-01-01 07:12:00', End='1970-01-01 16:04:30', Day

In [225]:
df = pd.DataFrame([
    dict(Start='1970-01-01 11:40:00', End='1970-01-01 14:47:30', Day='32'),
dict(Start='1970-01-01 11:24:00', End='1970-01-01 18:07:30', Day='31'),
dict(Start='1970-01-01 20:55:00', End='1970-01-01 04:53:00', Day='31'),
dict(Start='1970-01-01 12:59:00', End='1970-01-01 03:08:00', Day='30'),
dict(Start='1970-01-01 10:18:00', End='1970-01-01 18:01:30', Day='28'),
dict(Start='1970-01-01 10:24:30', End='1970-01-01 18:06:30', Day='27'),
dict(Start='1970-01-01 14:07:00', End='1970-01-01 18:30:30', Day='26'),
dict(Start='1970-01-01 21:44:00', End='1970-01-01 07:49:30', Day='26'),
dict(Start='1970-01-01 20:08:00', End='1970-01-01 03:35:00', Day='25'),
dict(Start='1970-01-01 11:50:30', End='1970-01-01 17:59:00', Day='23'),
dict(Start='1970-01-01 12:17:30', End='1970-01-01 21:17:30', Day='22'),
dict(Start='1970-01-01 02:18:30', End='1970-01-01 16:38:30', Day='21'),
dict(Start='1970-01-01 09:18:30', End='1970-01-01 18:37:30', Day='20'),
dict(Start='1970-01-01 07:12:00', End='1970-01-01 16:04:30', Day='19'),
dict(Start='1970-01-01 09:44:00', End='1970-01-01 18:11:30', Day='18'),
dict(Start='1970-01-01 11:12:00', End='1970-01-01 18:33:30', Day='17'),
dict(Start='1970-01-01 07:43:00', End='1970-01-01 18:05:30', Day='16'),
dict(Start='1970-01-01 10:05:30', End='1970-01-01 19:05:30', Day='15'),
dict(Start='1970-01-01 09:22:30', End='1970-01-01 17:44:30', Day='14'),
dict(Start='1970-01-01 07:32:00', End='1970-01-01 15:25:30', Day='13'),
dict(Start='1970-01-01 00:46:00', End='1970-01-01 10:56:30', Day='12'),
dict(Start='1970-01-01 02:48:00', End='1970-01-01 12:04:00', Day='11'),
dict(Start='1970-01-01 10:00:00', End='1970-01-01 16:03:30', Day='10'),
dict(Start='1970-01-01 11:16:00', End='1970-01-01 16:24:00', Day='9'),
dict(Start='1970-01-01 05:23:00', End='1970-01-01 16:04:00', Day='8'),
dict(Start='1970-01-01 19:44:00', End='1970-01-01 07:06:30', Day='7'),
dict(Start='1970-01-01 01:35:00', End='1970-01-01 08:55:30', Day='6'),
dict(Start='1970-01-01 10:00:30', End='1970-01-01 14:25:00', Day='5'),
dict(Start='1970-01-01 09:40:30', End='1970-01-01 13:49:30', Day='4'),
dict(Start='1970-01-01 12:55:00', End='1970-01-01 21:03:30', Day='3'),
dict(Start='1970-01-01 15:47:00', End='1970-01-01 03:55:30', Day='3'),
dict(Start='1970-01-01 20:40:00', End='1970-01-01 07:12:30', Day='2'),
dict(Start='1970-01-01 23:00:00', End='1970-01-01 03:58:00', Day='1'),
])

title = 'Timeline'

fig = go.Figure()
fig = px.timeline(df, x_start="Start", x_end="End", y='Day')
fig.update_xaxes(
    tickformat="%H\n%M",
    tickformatstops=[
        dict(dtickrange=[0, 86399999], value="%H:%M")]  # range is 1 hour to 24 hours
)

# fig.update_yaxes(autorange="reversed") # otherwise tasks are listed from the bottom up

# fig.update_layout(xaxis=dict(
#                       title='Timestamp', 
#                       tickformat = '%H:%M:%S',
#                       range=['2020-01-01 00:00:00','2020-01-01 23:59:59']
#                   ),
#                   yaxis=dict(
#                       title='Day',
#                       tickformat = '%m/%d/%Y'
#                    ))


app = jupyter_dash.JupyterDash(__name__,
                               external_stylesheets=external_stylesheets,
                               title=f"{title}")

app.layout = html.Div([
    dcc.Graph(
        figure=fig,
        
        #https://plotly.com/python/setting-graph-size/
        #https://stackoverflow.com/questions/46287189/how-can-i-change-the-size-of-my-dash-graph
        style={'height': '95vh'}
    ),
])

if __name__ == '__main__':
    app.run_server(mode='external', port=9000, debug=True)

Dash app running on http://127.0.0.1:9000/


In [ ]:
sleep_df.to_csv('data/sleep.csv', index=None, encoding='utf-8')